# Experiment Runner with Automatic Data Loading

This notebook demonstrates the ExperimentRunner which:
1. **Loads experiment configuration from YAML** - including data loading config
2. **Automatically loads and parses data** - using configured loader and parser
3. **Runs multiple RAG configs** - either sequentially or in parallel
4. **Generates detailed reports** - per-query and aggregate metrics
5. **Compares results** - across different configurations

The experiment configuration in YAML specifies:
- Data source (HuggingFace dataset)
- Data parser (e.g., title_passage)
- RAG config directory
- Number of queries to evaluate
- Parallel execution settings

## 1. Setup & Dependencies

In [1]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 2. Imports

In [2]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Change to project root directory
os.chdir(Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd())
project_root = Path.cwd()

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner

load_dotenv(override=True)
print("Current directory:", Path.cwd())
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

/Users/adityanarayan/git/rag-foundry/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current directory: /Users/adityanarayan/git/rag-foundry
HuggingFace token loaded: True
Groq API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [3]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/pubmedqa_experiment.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/pubmedqa-experiment/config
  Report Dir:  rag-experiments/pubmedqa-experiment/reports
  Temp Dir:    rag-experiments/pubmedqa-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache'}
  Num Queries: 20
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'pubmedqa', 'split': 'test', 'limit': 2450}

Data Parser:
  Type: {'type': 'noop'}


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [4]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.sections[0].per_query)
    
    # Display aggregate statistics
    print("\nAggregate Statistics:")
    display(report.sections[0].summary)

In [5]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/pubmedqa (test)...
Loaded 2450 samples

✅ Data loaded successfully!
  Documents: 5932 parsed documents
  Raw Data:  2450 samples

First Sample:
  Question: Do Surface Porosity and Pore Size Influence Mechanical Properties and Cellular Response to PEEK?...
  Documents: 5


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [6]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f"Loaded {len(configs)} RAG pipeline configurations:")
for cfg in configs:
    print(f"  - {cfg.name}")
    print(f"    Chunking: {cfg.chunking.type.value}")
    print(f"    Retrieval Pipeline: ")
    for search in cfg.retrieval.search.searches:
        print(f"      - {search.type.value}")
    print(f"    Reranker: {cfg.retrieval.rerank.type.value}")
    print(f"    Generation: {cfg.generation.config['model']}")

Loaded 8 RAG pipeline configurations:
  - pubmedqa_title_aware_v11_query_transform
    Chunking: sentence
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile
  - pubmedqa_title_aware_v12_improved_prompt
    Chunking: sentence
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile
  - pubmedqa_title_aware_v13_balanced
    Chunking: sentence
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile
  - pubmedqa_title_aware_v14_pubmedbert_stepback
    Chunking: sentence
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile
  - pubmedqa_title_aware_v15_biomedical_reranker
    Chunking: sentence
    Retrieval Pipeline: 
      - dense
      - sparse
    Reranker: cross_encoder
    Generation: llama-3.3-70b-versatile
  - pubmedq

## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [7]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 8 configurations from 0 to 20 queries...
Parallel mode: False
Loading embedding model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8027.91it/s]


Loading reranker model: BAAI/bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5504.59it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 1sUsing key #0: ****fyaJUsing key #0: ****fyaJ

Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 2sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 2/20 (10.0%) | QPS: 0.66 | ETA: 27s | Elapsed: 3sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 3/20 (15.0%) | QPS: 0.75 | ETA: 23s | Elapsed: 4sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 4/20 (20.0%) | QPS: 0.80 | ETA: 20s | Elapsed: 5sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 5/20 (25.0%) | QPS: 0.83 | ETA: 18s | Elapsed: 6sUsing key #0: ****fyaJ
Progress: 5/20 (25.0%) | QPS: 0.71 | ETA: 21s | Elapsed: 7sUsing key #0: ****fyaJ
Progress: 6/20 (30.0%) | QPS: 0.75 | ETA: 19s | Elapsed: 8sUsing key #0: ****fyaJ
Using key #0: ****fyaJ
Progress: 7/20 (35.0%) | QPS: 0.77 | ETA: 17s | Elapsed: 9sUsing key 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57527.50it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #4: ****tmVo
Using key #4: ****tmVo
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 1sUsing key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Progress: 1/20 (5.0%) | QPS: 0.50 | ETA: 38s | Elapsed: 2sUsing key #4: ****tmVo
Using key #4: ****tmVo
Progress: 2/20 (10.0%) | QPS: 0.66 | ETA: 27s | Elapsed: 3sUsing key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Progress: 4/20 (20.0%) | QPS: 0.99 | ETA: 16s | Elapsed: 4sUsing key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Progress: 6/20 (30.0%) | QPS: 1.19 | ETA: 12s | Elapsed: 5sUsing key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Progress: 8/20 (40.0%) | QPS: 1.32 | ETA: 9s | Elapsed: 6sUsing key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Progress: 10/20 (50.0%) | QPS: 1.42 | ETA: 7s | Elapse

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 8141.71it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 1sUsing key #5: ****Z8YyUsing key #5: ****Z8Yy

Using key #5: ****Z8Yy
Progress: 1/20 (5.0%) | QPS: 0.50 | ETA: 38s | Elapsed: 2sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 3/20 (15.0%) | QPS: 0.99 | ETA: 17s | Elapsed: 3sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 5/20 (25.0%) | QPS: 1.24 | ETA: 12s | Elapsed: 4sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 6/20 (30.0%) | QPS: 1.19 | ETA: 12s | Elapsed: 5sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 8/20 (40.0%) | QPS: 1.33 | ETA: 9s | Elapsed: 6sUsing key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Using key #5: ****Z8Yy
Progress: 11/20 (55.0%) | QPS: 

## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [8]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

Found 8 configs with JSONL files
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
Using key #4: ****tmVo
429 on ****tmVo
Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kq4kd2eeekas3zmfykg6fw6d` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11499, Requested 1864. Please try again in 6.814999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Status: 429

Headers:
  date: Fri, 24 Jul 2026 18:24:45 GMT
  content-type: application/json
  content-length: 390
  connection: keep-alive
  cache-control: private, max-age=0, no-store, no-cache, must-revalidate
  retry-after: 7
  server: cloudflare
  vary: Origin
  x-groq-region: bom
  x-ratelimit-limit-

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [9]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

Generating reports...

✅ Reports generated!
  Saved to: rag-experiments/pubmedqa-experiment/reports


## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [10]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    


Configuration: pubmedqa_title_aware_v11_query_transform

Per-Query Results:


# RAG Multi-Config Evaluation Report

_Strategy: detailed_query_

## Config: `pubmedqa_title_aware_v11_query_transform`

**name**: pubmedqa_title_aware_v11_query_transform  •  **mode**: test  •  **providers**: {'groq': {'type': 'groq', 'api_key_env': 'GROQ_API_KEY', 'params': {'cooldown_seconds': 60}}}  •  **chunking**: {'type': 'sentence', 'config': {'max_words': 200, 'overlap_sentences': 1}}  •  **embedding**: {'type': 'sentence_transformer', 'config': {'model_name': 'BAAI/bge-large-en-v1.5', 'dimension': 1024}}  •  **vector_store**: {'type': 'faiss', 'config': {'dimension': 1024}}  •  **retrieval**: {'search': {'searches': [{'type': 'dense', 'config': {'top_k': 50, 'context_window': 1}}, {'type': 'sparse', 'config': {'top_k': 50}}]}, 'query_transform': {'type': 'multi_query', 'provider': 'groq', 'config': {'num_queries': 3, 'model': 'llama-3.3-70b-versatile', 'temperature': 0.3}}, 'fusion': {'type': 'rrf', 'config': {'k': 60}}, 'rerank': {'type': 'cross_encoder', 'config': {'model_name': 'BAAI/bge-reranker-v2-m3', 'top_k': 7}}}  •  **generation**: {'strategy': 'default', 'provider': 'groq', 'config': {'model': 'llama-3.3-70b-versatile', 'temperature': 0.0, 'max_tokens': 512, 'system_prompt': 'You are a biomedical question answering assistant.\nAnswer ONLY from the retrieved passages below.\nBe concise but complete — include all relevant details from the passages that address the question.\nDo NOT add information from your own knowledge.\nDo NOT hedge with phrases like "based on general knowledge" or "it is well-established".\nIf the passages do not contain enough information, say "The provided context does not contain sufficient information to answer this question."\n', 'user_prompt': 'Passages:\n{context}\n\nQuestion: {query}\n\nAnswer (from passages only, include all relevant details):\n'}}  •  **evaluation**: None  •  **cache**: {'enabled': True, 'cache_dir': './cache'}  •  **start_index**: None  •  **end_index**: None  •  **logging_config**: {'enabled': True, 'level': 'INFO', 'show_progress': True}

...

### Per-query results

,query,retrieved_documents,answer,relevance_score__pred,relevance_score__gt,relevance_score__deviation,utilization_score__pred,utilization_score__gt,utilization_score__deviation,completeness_score__pred,completeness_score__gt,completeness_score__deviation,adherence_score__pred,adherence_score__gt,adherence_score__deviation
0,Do Surface Porosity and Pore Size Influence Mechanical Properties and Cellul...,"[{'text': '', 'title': '', 'content': 'Despite its widespread use in orthopa...","According to the passages, surface porosity and pore size do influence the m...",0.8235,0.6111,0.2124,0.4118,0.5000,-0.0882,0.5000,0.7273,-0.2273,0.0,1.0,-1.0
1,Does prosthetic covering of nitinol stents alter healing characteristics or ...,"[{'text': '', 'title': '', 'content': 'To evaluate placement of polyester (D...",The passages provide information on the evaluation of prosthetic coverings o...,0.5556,0.5000,0.0556,0.3333,0.4000,-0.0667,0.6000,0.8000,-0.2000,0.0,0.0,0.0
2,Is diabetes mellitus a negative prognostic factor for the treatment of advan...,"[{'text': '', 'title': '', 'content': 'It has been demonstrated that there a...",The passages do not provide sufficient information to answer this question.,0.3636,0.2857,0.0779,0.0909,0.1429,-0.0520,0.2500,0.5000,-0.2500,1.0,1.0,0.0
3,Do elderly persons need to be encouraged to drink more fluids?,"[{'text': '', 'title': '', 'content': 'Dehydration in older adults contribut...",The passages do not provide sufficient information to answer this question.,0.3333,0.7692,-0.4359,0.0000,0.4615,-0.4615,0.0000,0.6000,-0.6000,1.0,1.0,0.0
4,Do the economic and social factors play an important role in relation to the...,"[{'text': '', 'title': '', 'content': 'To evaluate the behavior of contact l...",The passages do not provide sufficient information to answer this question.,0.0909,0.7000,-0.6091,0.0000,0.1000,-0.1000,0.0000,0.1429,-0.1429,1.0,1.0,0.0
5,Are prostate biopsies mandatory in patients with prostate-specific antigen i...,"[{'text': '', 'title': '', 'content': 'Aim of this study was to evaluate if ...",The passages suggest that prostate biopsies may not be mandatory in patients...,0.6667,0.5000,0.1667,0.4444,0.2500,0.1944,0.6667,0.5000,0.1667,0.0,1.0,-1.0
6,Do working conditions explain the increased risks of disability pension amon...,"[{'text': '', 'title': '', 'content': 'Rates of disability pension are great...","According to the passages, working conditions do contribute to explaining th...",0.5000,0.4615,0.0385,0.2857,0.1538,0.1319,0.5714,0.3333,0.2381,0.0,1.0,-1.0
7,Platelet aggregation according to body mass index in patients undergoing cor...,"[{'text': '', 'title': '', 'content': 'A 300 mg clopidogrel loading-dose (LD...",The passages suggest that the relationship between body mass index (BMI) and...,0.7143,0.3750,0.3393,1.0000,0.1875,0.8125,1.0000,0.5000,0.5000,0.0,1.0,-1.0
8,Interleukin 27 polymorphisms in HCV RNA positive patients: is there an impac...,"[{'text': '', 'title': '', 'content': 'Interleukin 27 (IL-27) has pleiotropi...",The passages provide some information about the relationship between interle...,0.7000,0.5333,0.1667,0.6000,0.5333,0.0667,0.8571,1.0000,-0.1429,0.0,1.0,-1.0
9,Can the Functional Movement Screen™ be used to capture changes in spine and ...,"[{'text': '', 'title': '', 'content': 'To examine whether objective measures...",The Functional Movement Screen (FMS) can be used to capture changes in spine...,0.8571,0.5714,0.2857,0.7143,0.4286,0.2857,0.8333,0.7500,0.0833,0.0,0.0,0.0


### Aggregate TRACe scores (mean / ground truth / deviation)

,metric,mean_score,mean_ground_truth,std_score,std_ground_truth,mean_abs_error
0,relevance_score,0.4980,0.4885,0.2196,0.2032,0.2161
1,utilization_score,0.3706,0.3699,0.2995,0.2343,0.2709
2,completeness_score,0.5498,0.6849,0.3691,0.2407,0.3566
3,adherence_score,0.3000,0.8500,0.4583,0.3571,0.6500


None

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [11]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

Generating comparison report...

✅ Comparison report generated!
  Saved to: rag-experiments/pubmedqa-experiment/reports/comparison.csv


In [12]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())


Configuration Comparison:


,config_name,relevance_score__mean,relevance_score__mae,utilization_score__mean,utilization_score__mae,completeness_score__mean,completeness_score__mae,adherence_score__mean,adherence_score__mae
0,pubmedqa_title_aware_v11_query_transform,0.5317,0.1724,0.2653,0.1785,0.5146,0.3393,0.20,0.65
1,pubmedqa_title_aware_v14_pubmedbert_stepback,0.3471,0.2087,0.1186,0.2625,0.2617,0.4767,0.50,0.55
2,pubmedqa_title_aware_v15_biomedical_reranker,0.3441,0.2426,0.1534,0.2688,0.3221,0.4758,0.65,0.50
3,pubmedqa_title_aware_v16_improved_retrieval,0.4815,0.2007,0.4270,0.2900,0.6695,0.3462,0.15,0.70
4,pubmedqa_title_aware_v18_balanced,0.4980,0.2161,0.3706,0.2709,0.5498,0.3566,0.30,0.65
5,pubmedqa_title_aware_v13_balanced,0.5898,0.1904,0.2970,0.1718,0.5330,0.2647,0.20,0.65
6,pubmedqa_title_aware_v12_improved_prompt,0.4747,0.1818,0.2242,0.2174,0.3458,0.4420,0.45,0.60
7,pubmedqa_title_aware_v17_hybrid_transform,0.4610,0.2491,0.2410,0.2619,0.3828,0.4098,0.45,0.50


## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons